# 二次分配问题 (QAP)

**类别：** 选址

来源： [https://www.hexaly.com/templates/quadratic-assignment-problem-qap](https://www.hexaly.com/templates/quadratic-assignment-problem-qap)


## 问题

**在 Quadratic Assignment Problem (QAP) 中**，需要将 n 个设施分配到 n 个位置。问题数据包含每对位置之间的距离以及每对设施之间的流量（或权重），即它们之间运输的物料量。问题目标是将每个设施分配到一个位置，使距离与对应流量乘积之和最小。该问题类似于指派问题（Assignment Problem），不同之处在于其目标函数由二次不等式表示，因此得名。更多细节，请参阅 [QAPLIB 网页](https://coral.ise.lehigh.edu/data-sets/qaplib/qaplib-problem-instances-and-solutions/)。

	

### 学到的建模原则

- 添加 [list decision variable](https://www.hexaly.com/docs/last/mathematicaloperators/collectionvariables.html) 来建模设施的排列
- 使用 [‘count’ operator](https://www.hexaly.com/docs/last/mathematicaloperators/collectionvariables.html#unary-and-binary-operators) 约束 list 中的元素数量
- 使用 [‘at’ operator](https://www.hexaly.com/docs/last/mathematicaloperators/collectionvariables.html#operators-specific-to-lists) 访问 list 中的元素以定义目标函数


## 数据

我们提供来自 [QAPLIB](http://anjos.mgi.polymtl.ca/qaplib/) 的实例。数据文件的格式如下：

- 点的数量
- 矩阵 A：每对位置之间的距离
- 矩阵 B：每对设施之间的流量


## 模型

Quadratic Assignment Problem (QAP) 的 Hexaly 模型仅使用一个 [list decision variable](https://www.hexaly.com/docs/last/mathematicaloperators/collectionvariables.html)。该 list 表示设施的一个排列：list 中第 i 个位置的元素表示分配到位置 i 的设施索引。

使用 **count** 算子，我们将 list 的大小约束为等于设施总数。从而确保所有设施都被分配到一个位置。

最后，我们计算目标函数。使用 **at** 算子，我们可以访问分配到位置 i 的设施（p[i]，其中 p 是 list 变量）。然后我们可以轻松地获取分配到位置 i 和 j 的设施之间的流量（B[p[i]][p[j]]）。将该数量乘以位置 i 和 j 之间的距离（A[i][j]），即可得到与这两个设施相关联的成本。


## Results

在 QAPLIB 研究基准上，对于最多 **256 个设施** 的实例，Hexaly Optimizer 在 1 分钟运行时间内对 Quadratic Assignment Problem (QAP) 达到了 **1.1% 的平均最优性差距**。我们的 [Quadratic Assignment Problem (QAP) benchmark page](https://www.hexaly.com/benchmark/hexaly-vs-gurobi-quadratic-assignment-problem) 展示了 Hexaly Optimizer 在这一具有挑战性的问题上如何超越 Gurobi 等传统通用优化求解器。

[Explore this benchmark](https://www.hexaly.com/benchmark/hexaly-vs-gurobi-quadratic-assignment-problem)


## Python 实现


In [ ]:
# Copyright (c) Hexaly. Permission is hereby granted to use, copy,
# and modify this code for applications developed with Hexaly.
import hexaly.optimizer
import sys

if len(sys.argv) < 2:
    print("Usage: python qap.py inputFile [outputFile] [timeLimit]")
    sys.exit(1)


def read_integers(filename):
    with open(filename) as f:
        return [int(elem) for elem in f.read().split()]


with hexaly.optimizer.HexalyOptimizer() as optimizer:
    #
    # Read instance data
    #
    file_it = iter(read_integers(sys.argv[1]))

    # Number of points
    n = next(file_it)

    # Distance between locations
    A = [[next(file_it) for j in range(n)] for i in range(n)]
    # Flow between factories
    B = [[next(file_it) for j in range(n)] for i in range(n)]

    #
    # Declare the optimization model
    #
    model = optimizer.model

    # Permutation such that p[i] is the facility on the location i
    p = model.list(n)

    # The list must be complete
    model.constraint(model.eq(model.count(p), n))

    # Create B as an array to be accessed by an at operator
    array_B = model.array(B)

    # Minimize the sum of product distance*flow
    obj = model.sum(A[i][j] * model.at(array_B, p[i], p[j])
                    for j in range(n) for i in range(n))
    model.minimize(obj)

    model.close()

    # Parameterize the optimizer
    if len(sys.argv) >= 4:
        optimizer.param.time_limit = int(sys.argv[3])
    else:
        optimizer.param.time_limit = 30
    optimizer.solve()

    #
    # Write the solution in a file with the following format:
    #  - n objValue
    #  - permutation p
    #
    if len(sys.argv) >= 3:
        with open(sys.argv[2], 'w') as outfile:
            outfile.write("%d %d\n" % (n, obj.value))
            for i in range(n):
                outfile.write("%d " % p.value[i])
            outfile.write("\n")
